# 02 Data Cleaning

This notebook cleans the merged dataset and defines key variables
for subsequent analysis.

Input: merged_data.csv (from notebook 01)
Output: cleaned_data.csv (used in notebooks 03 and 04)

In [7]:
import pandas as pd

# Load merged dataset from notebook 01
df = pd.read_csv('../data/merged_data.csv')

In [8]:
# Check missing values before cleaning
# PFQ061D: 2766 missing due to NHANES questionnaire skip pattern
print(df.isnull().sum())
print("Total rows:", df.shape[0])

SEQN           0
RIDAGEYR       0
RIAGENDR       0
BMXBMI        99
PAD680        10
PAQ605         0
PFQ061D     2766
dtype: int64
Total rows: 5533


In [9]:
print("Initial sample:", df.shape[0])

# Filter to working-age adults (20–65) to reduce age-related confounding
df = df[(df['RIDAGEYR'] >= 20) & (df['RIDAGEYR'] <= 65)]
print("After age filter:", df.shape[0])

# Remove invalid sedentary time values (9999 = don't know)
df = df[df['PAD680'] != 9999]
print("After sedentary cleaning:", df.shape[0])

# Remove missing and invalid physical activity values (7=refused, 9=don't know)
df = df[df['PAQ605'].notna()]
df = df[~df['PAQ605'].isin([7, 9])]
print("After activity cleaning:", df.shape[0])

# Remove missing and invalid functional limitation values (5=don't do, 7=refused, 9=don't know)
df = df[df['PFQ061D'].notna()]
df = df[~df['PFQ061D'].isin([5, 7, 9])]
print("After functional limitation cleaning:", df.shape[0])

# Complete-case analysis: imputation avoided as BMI is a primary predictor
df = df[df['BMXBMI'].notna()]
print("After BMI cleaning:", df.shape[0])

# Convert to hours; remove implausible values (extreme values likely reflect reporting errors)
df['sitting_hours'] = df['PAD680'] / 60
df = df[(df['sitting_hours'] >= 0.5) & (df['sitting_hours'] <= 18)]
print("Final sample:", df.shape[0])

Initial sample: 5533
After age filter: 3962
After sedentary cleaning: 3941
After activity cleaning: 3937
After functional limitation cleaning: 1424
After BMI cleaning: 1402
Final sample: 1374


The large reduction in sample size is primarily driven by missing 
PFQ061D responses, as functional limitation questions were only 
administered to eligible respondents under NHANES skip patterns.
Complete-case analysis may introduce selection bias; this is 
acknowledged as a study limitation.

In [10]:
# PFQ061D: 1=no difficulty, 2=some, 3=much, 4=unable; binarized as 0/1
df['functional_limitation'] = (df['PFQ061D'] >= 2).astype(int)

# PAQ605 measures vigorous recreational activity (1=Yes, 2=No)
# refused/don't know already excluded above
df['vigorous_activity'] = (df['PAQ605'] == 1).astype(int)

print(df['functional_limitation'].value_counts())
print(df['vigorous_activity'].value_counts())

functional_limitation
1    783
0    591
Name: count, dtype: int64
vigorous_activity
0    1028
1     346
Name: count, dtype: int64


In [11]:
# Summary statistics of key continuous variables after cleaning
print("Final sample size:", df.shape[0])
print(df[['RIDAGEYR', 'BMXBMI', 'sitting_hours']].describe())

Final sample size: 1374
          RIDAGEYR       BMXBMI  sitting_hours
count  1374.000000  1374.000000    1374.000000
mean     53.039301    31.133406       5.605046
std      12.242449     8.233689       3.425961
min      20.000000    14.800000       0.500000
25%      46.000000    25.525000       3.000000
50%      59.000000    29.700000       5.000000
75%      62.000000    35.100000       8.000000
max      65.000000    74.800000      18.000000


In [12]:
# Save cleaned data for use in notebook 03
df.to_csv('../data/cleaned_data.csv', index=False)
print("Cleaned data saved.")

Cleaned data saved.
